# Exception Contract Audit (ECA-6)
Jacek Adamek. Local review notebook; no publication or inference is triggered.
This notebook reads recorded results. Optional full test reproduction is described below.

In [1]:
from pathlib import Path
import json, hashlib
project = Path.cwd()
resource = project / "results/exception-contract-resource"
read = lambda name: json.loads((resource/name).read_text())
protocol = read("protocol.json")
assert hashlib.sha256((project/"contract_resource.py").read_bytes()).hexdigest() == protocol["generator_sha256"]
print("Original tasks:", len(protocol["cases"]), "| calibration patches:", sum(len(c["mutants"]) for c in protocol["cases"]))

Original tasks: 6 | calibration patches: 36


In [2]:
calibration = read("calibration-evaluation.json")
model = read("model-evaluation.json")
v2 = read("posthoc-v2-evaluation.json")
print("V1 calibration accepted by weak/contract:", calibration["weak_pass_count"], calibration["contract_pass_count"])
print("Model accepted by weak/v1:", model["weak_pass_count"], model["contract_pass_count"])
print("Post-outcome v2: model accepted", v2["model_passes"], "; calibration rejected", v2["mutants_rejected"])
assert model["weak_pass_count"] == model["contract_pass_count"] == 6
assert v2["model_passes"] == 5 and v2["mutants_rejected"] == 36

V1 calibration accepted by weak/contract: 42 7
Model accepted by weak/v1: 6 6
Post-outcome v2: model accepted 5 ; calibration rejected 36


In [3]:
for row in v2["cases"]:
    if row["variant"] == "model" and not row["passed"]:
        print(row["id"], row["failed_tests"])
real = json.loads((project/"results/itsdangerous-model-budget1024/payload-diagnostic.json").read_text())
for row in real["rows"]:
    print(row["variant"], row["exception"], row["payload_type"], row["payload_repr"])

type_tuple ['test_existing_negative_error_message']
fixed BadTimeSignature bytes b'value'
parent OSError NoneType None
generated BadTimeSignature str 'Timestamp 21959264258072993 caused OSError: [Errno 22] Invalid argument'


## Reproduction and boundaries
Install pytest 8.4.2, then run `python reproduce_contract_resource.py FRESH_DESTINATION` to recompute all resource test outcomes without Gemma. Model inference requires the separately acquired pinned weights/runtime.
The six tasks are related constructed diagnostics. V2 is post-outcome development, not independent validation. Read ECA-6-DATASHEET.txt and submission-writeup.txt.
The executed copy uses explicit plain CPython execution of code cells, not a Jupyter-kernel claim.